# UnifyWeaver-এ উন্নত রিকার্শন প্যাটার্ন

এই নোটবুকটি চারটি প্রধান রিকার্শন প্যাটার্ন প্রদর্শন করে যা UnifyWeaver শনাক্ত এবং অপ্টিমাইজ করতে পারে:

1. **টেইল রিকার্শন (Tail Recursion)** - অ্যাকুমুলেটর সহ পুনরাবৃত্তিমূলক লুপ
2. **লিনিয়ার রিকার্শন (Linear Recursion)** - মেমোজাইজেশন সহ একক রিকার্সিভ কল
3. **ট্রি রিকার্শন (Tree Recursion)** - কাঠামোর বিভিন্ন অংশের ওপর একাধিক রিকার্সিভ কল
4. **মিউচুয়াল রিকার্শন (Mutual Recursion)** - চক্রাকারে একে অপরকে কল করা প্রেডিকেটসমূহ

## শেখার উদ্দেশ্য

- বিভিন্ন রিকার্শন প্যাটার্ন বোঝা
- দেখা কীভাবে UnifyWeaver প্রতিটি প্যাটার্ন শনাক্ত ও অপ্টিমাইজ করে
- কর্মক্ষমতার বৈশিষ্ট্য তুলনা করা
- কখন কোন প্যাটার্ন ব্যবহার করতে হবে তা শেখা

## সেটআপ

UnifyWeaver পরিবেশ শুরু করুন।

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## প্যাটার্ন ১: টেইল রিকার্শন (Tail Recursion)

টেইল রিকার্শন মধ্যবর্তী ফলাফল এগিয়ে নিতে একটি অ্যাকুমুলেটর ব্যবহার করে এবং রিকার্সিভ কলটি ফাংশনের **শেষ ক্রিয়া** হিসেবে থাকে।

### উদাহরণ: তালিকার উপাদান গণনা করা

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### Prolog-এ পরীক্ষা করুন

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### প্যাটার্ন সনাক্তকরণ পরীক্ষা করুন

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Bash-এ কম্পাইল করুন

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### জেনারেট করা Bash পরীক্ষা করুন

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## প্যাটার্ন ২: লিনিয়ার রিকার্শন (Linear Recursion)

লিনিয়ার রিকার্শনে প্রতি ক্লজে **ঠিক একটি** রিকার্সিভ কল থাকে, এবং রিকার্সিভ কল ফিরে আসার পরে গণনা সম্পন্ন হয়।

### উদাহরণ: ফ্যাক্টোরিয়াল (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### Prolog-এ পরীক্ষা করুন

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### প্যাটার্ন সনাক্তকরণ পরীক্ষা করুন

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Bash-এ কম্পাইল করুন

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### জেনারেট করা Bash পরীক্ষা করুন

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## প্যাটার্ন ৩: ট্রি রিকার্শন (Tree Recursion)

কাঠামোর বিভিন্ন অংশ প্রসেস করার জন্য ট্রি রিকার্শন **একাধিক** রিকার্সিভ কল সম্পাদন করে।

### উদাহরণ: ট্রি যোগফল (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### Prolog-এ পরীক্ষা করুন

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Bash-এ কম্পাইল করুন

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### জেনারেট করা Bash পরীক্ষা করুন

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## প্যাটার্ন ৪: মিউচুয়াল রিকার্শন (Mutual Recursion)

মিউচুয়াল রিকার্শন তখনই ঘটে যখন দুই বা ততোধিক প্রেডিকেট একটি চক্রের মধ্যে একে অপরকে কল করে।

### উদাহরণ: জোড় (Even) এবং বিজোড় (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### Prolog-এ পরীক্ষা করুন

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### মিউচুয়াল রিকার্শন পরীক্ষা করুন

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Bash-এ কম্পাইল করুন

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### জেনারেট করা Bash পরীক্ষা করুন

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## প্যাটার্ন তুলনা

আসুন প্রতিটি প্যাটার্নের বৈশিষ্ট্য তুলনা করি:

| প্যাটার্ন | রিকার্সিভ কল | অপ্টিমাইজেশন | স্পেস জটিলতা | যার জন্য সবচেয়ে উপযুক্ত |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **টেইল** | ১টি (টেইল অবস্থানে) | পুনরাবৃত্তিমূলক লুপ | O(1) | অ্যাকুমুলেটর, লিনিয়ার স্ক্যান |
| **লিনিয়ার** | ১টি (যেকোনো অবস্থানে) | ফোল্ড + মেমোজাইজেশন | O(n) মেমো টেবিল | ফিবোনাচ্চি, ফ্যাক্টোরিয়াল |
| **ট্রি** | ২+ (কাঠামোর অংশ) | কাঠামোগত বিভাজন | O(গভীরতা) স্ট্যাক | ট্রি/গ্রাফ অপারেশন |
| **মিউচুয়াল** | ১+ (প্রেডিকেটের মধ্যে) | শেয়ার্ড মেমোজাইজেশন | O(n) শেয়ার্ড টেবিল | জোড়/বিজোড়, পারস্পরিক সংজ্ঞা |

## প্যাটার্ন সনাক্তকরণের ক্রম

UnifyWeaver এই ক্রমে প্যাটার্ন মেলানোর চেষ্টা করে:

1. **টেইল রিকার্শন** (সবচেয়ে কার্যকর)
2. **লিনিয়ার রিকার্শন** (যদি নিষিদ্ধ না হয়)
3. **ট্রি রিকার্শন** (কাঠামোগত)
4. **মিউচুয়াল রিকার্শন** (SCC সনাক্তকরণ)
5. **বেসিক রিকার্শন** (ডিফল্ট ফলব্যাক)

আপনি `forbid_linear_recursion/1` দিয়ে সনাক্তকরণ প্রক্রিয়া প্রভাবিত করতে পারেন।

## অনুশীলন: আপনার পালা!

এই প্রেডিকেটগুলো সংজ্ঞায়িত এবং কম্পাইল করার চেষ্টা করুন:

### ১. টেইল রিকার্সিভ যোগফল
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### ২. লিনিয়ার রিকার্সিভ ফিবোনাচ্চি
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### ৩. ট্রির উচ্চতা
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## সারসংক্ষেপ

এই নোটবুকে আপনি শিখেছেন:

✅ UnifyWeaver-এর চারটি প্রধান রিকার্শন প্যাটার্ন

✅ Prolog-এ প্রতিটি প্যাটার্ন কীভাবে সংজ্ঞায়িত করতে হয়

✅ UnifyWeaver কীভাবে প্রতিটি প্যাটার্ন শনাক্ত ও অপ্টিমাইজ করে

✅ প্রতিটি প্যাটার্নের কর্মক্ষমতার বৈশিষ্ট্য

✅ কখন কোন প্যাটার্ন ব্যবহার করতে হবে

## পরবর্তী পদক্ষেপ

উন্নত কোড বিশ্লেষণ এবং ভিজ্যুয়ালাইজেশন সম্পর্কে জানতে **নোটবুক ৩: কল গ্রাফ ভিজ্যুয়ালাইজেশন**-এ এগিয়ে যান!